In [1]:
# %% 初始化环境
import sys
sys.path.append("..")
from experiments.environment import *
init_environment()

Applied font.family: ['Times New Roman', 'Microsoft YaHei', 'DejaVu Sans']
findfont Times -> C:\Windows\Fonts\times.ttf
findfont CN -> C:\Windows\Fonts\msyh.ttc
🚀 环境初始化完成。


In [2]:
from core.data_processor import TabularDataProcessor

In [3]:
import pandas as pd
import numpy as np

# 如果你的 TabularDataProcessor 已经定义在同一个 notebook 中，则直接运行类定义即可。
# 如果保存为 core/data_processor.py，则使用：
# from core.data_processor import TabularDataProcessor

# 为了演示，这里假设类已经定义在上一个 cell 中（例如执行了你的类代码）

def create_test_data(seed=42):
    np.random.seed(seed)
    n = 50

    age = np.random.randint(18, 90, n).astype(float)
    age[np.random.choice(n, 10, replace=False)] = np.nan

    bmi = np.random.normal(25, 3, n).clip(15, 40)
    bmi = np.round(bmi, 1)
    bmi[np.random.choice(n, 15, replace=False)] = np.nan

    sys_bp = np.random.choice([115, 120.5, 120, 121.5, 122, 117.5], n)
    sys_bp = sys_bp.astype(float)
    sys_bp[np.random.choice(n, 12, replace=False)] = np.nan

    edu = np.random.choice(['low', 'medium', 'high'], n)
    edu[np.random.choice(n, 8, replace=False)] = None

    return pd.DataFrame({'age': age, 'bmi': bmi, 'sys_bp': sys_bp, 'edu': edu})

df = create_test_data()
print("原始数据前5行：")
display(df.head(10))

原始数据前5行：


,age,bmi,sys_bp,edu
0,69.0,22.5,120.0,low
1,32.0,24.1,121.5,medium
2,NaN,26.0,121.5,None
3,78.0,27.9,115.0,low
4,NaN,23.6,115.0,low
5,41.0,24.4,120.5,low
6,20.0,NaN,115.0,low
7,39.0,NaN,120.0,medium
8,70.0,NaN,NaN,None
9,19.0,NaN,NaN,high


In [4]:
import pandas as pd
import numpy as np

# 假设你的 TabularDataProcessor 类已经在当前 cell 中定义
# 如果不在，请先执行类定义的 cell。

def create_test_data(seed=42):
    np.random.seed(seed)
    n = 200
    age = np.random.randint(18, 90, n).astype(float)
    age[np.random.choice(n, 10, replace=False)] = np.nan
    bmi = np.random.normal(25, 3, n).clip(15, 40)
    bmi = np.round(bmi, 1)
    bmi[np.random.choice(n, 15, replace=False)] = np.nan
    sys_bp = np.random.choice([120, 120.5, 121, 121.5, 122, 122.5], n)
    sys_bp = sys_bp.astype(float)
    sys_bp[np.random.choice(n, 12, replace=False)] = np.nan
    edu = np.random.choice(['low', 'medium', 'high'], n)
    edu[np.random.choice(n, 8, replace=False)] = None
    return pd.DataFrame({'age': age, 'bmi': bmi, 'sys_bp': sys_bp, 'edu': edu})

df = create_test_data()

def test_pipeline(num_method, cat_method):
    processor = TabularDataProcessor(
        num_cols=['age', 'bmi', 'sys_bp'],
        cat_cols=['edu'],
        num_transform_method=num_method,
        cat_encoding_method=cat_method,
        random_state=42
    )
    
    # fit & transform
    processor.fit(df)
    transformed = processor.transform(df)
    print(f"[{num_method} / {cat_method}] transform 输出形状: {transformed.shape}")
    
    # 对于 label 模式，验证逆变换一致性
    if cat_method == 'label':
        reconstructed = processor.inverse_transform(transformed)
        # 验证数值列
        for col in processor.num_cols:
            orig = df[col].dropna()
            recon = reconstructed.loc[orig.index, col]
            meta = processor.meta[col]
            if meta['is_int']:
                assert (recon == orig).all(), f"整数列 {col} 还原失败"
            else:
                if meta['quantize_step'] is not None:
                    assert np.allclose(recon, orig, atol=meta['quantize_step']/2), f"半整数列 {col} 误差过大"
                else:
                    assert np.allclose(recon, orig, atol=1e-2), f"连续列 {col} 误差过大"
        # 验证类别列
        for col in processor.cat_cols:
            orig = df[col].dropna()
            recon = reconstructed.loc[orig.index, col]
            assert (recon == orig).all(), f"类别列 {col} 还原失败"
        print(f"✓ 逆变换验证通过")
    else:
        # none 模式：仅检查 transform 输出中类别列是否为字符串/object
        assert transformed['edu'].dtype == object, "none 模式下类别列应返回原始字符串"
        print(f"✓ transform 输出正确（类别列为原始字符串）")
    
    print(f"===== 测试完成 [{num_method} / {cat_method}] =====\n")

# 运行所有参数组合
for nm in ['quantile', 'standard', 'none']:
    for cm in ['label', 'none']:
        test_pipeline(nm, cm)

[quantile / label] transform 输出形状: (200, 4)
✓ 逆变换验证通过
===== 测试完成 [quantile / label] =====

[quantile / none] transform 输出形状: (200, 4)
✓ transform 输出正确（类别列为原始字符串）
===== 测试完成 [quantile / none] =====

[standard / label] transform 输出形状: (200, 4)
✓ 逆变换验证通过
===== 测试完成 [standard / label] =====

[standard / none] transform 输出形状: (200, 4)
✓ transform 输出正确（类别列为原始字符串）
===== 测试完成 [standard / none] =====

[none / label] transform 输出形状: (200, 4)
✓ 逆变换验证通过
===== 测试完成 [none / label] =====

[none / none] transform 输出形状: (200, 4)
✓ transform 输出正确（类别列为原始字符串）
===== 测试完成 [none / none] =====



In [5]:
processor = TabularDataProcessor(
    num_cols=['age', 'bmi', 'sys_bp'],
    cat_cols=['edu'],
    num_transform_method='quantile',
    cat_encoding_method='none',
    random_state=42
)
processor.fit(df)
transformed = processor.transform(df)
print("类别列 dtype:", transformed['edu'].dtype)
print("前5个值：", transformed['edu'].head().tolist())

类别列 dtype: object
前5个值： ['high', 'high', 'low', 'low', 'low']


In [6]:
processor = TabularDataProcessor(
    num_cols=['age', 'bmi', 'sys_bp'],
    cat_cols=[],
    num_transform_method='quantile'
)
processor.fit(df[['age', 'bmi', 'sys_bp']])
transformed = processor.transform(df[['age', 'bmi', 'sys_bp']])
reconstructed = processor.inverse_transform(transformed)
print("只有数值列：还原后与原始数据比较（非缺失值）")
for col in processor.num_cols:
    orig = df[col].dropna()
    recon = reconstructed.loc[orig.index, col]
    assert np.allclose(recon, orig, atol=1e-2, equal_nan=True)
print("✓ 通过")

只有数值列：还原后与原始数据比较（非缺失值）
✓ 通过


In [ ]:
processor = TabularDataProcessor(
    num_cols=[],
    cat_cols=['edu'],
    cat_encoding_method='label'
)
processor.fit(df[['edu']])
transformed = processor.transform(df[['edu']])
reconstructed = processor.inverse_transform(transformed)
assert (reconstructed['edu'] == df['edu']).all()
print("✓ 只有类别列测试通过")

In [ ]:
processor = TabularDataProcessor(
    num_cols=['age', 'bmi', 'sys_bp'],
    cat_cols=['edu'],
    num_transform_method='quantile'
)
processor.fit(df)
info = processor.get_model_info()
print("Model Info:")
for k, v in info.items():
    print(f"  {k}: {v}")

In [ ]:
# 创建一个临时实例（无需数据）
dummy = TabularDataProcessor(num_cols=[], cat_cols=[])

# 测试 _detect_quantize_step
uniq = np.array([1.0, 1.5, 2.0, 4.5])
step = dummy._detect_quantize_step(uniq, candidates=(0.5,))
print("半整数检测:", step)  # 应为 0.5
assert step == 0.5

uniq2 = np.array([1.0, 1.2, 1.4])
step2 = dummy._detect_quantize_step(uniq2, candidates=(0.5,))
print("非半整数检测:", step2)  # 应为 None

# 测试 _infer_precision
s = pd.Series([1.1, 2.2, 3.3, 4.4, 5.5, 1.111])
prec = dummy._infer_precision(s)
print(f"精度推断结果: {prec}")  # 可能是 1 或 2

print("所有辅助方法测试通过")

In [ ]:
processor = TabularDataProcessor(
    num_cols=['age', 'bmi', 'sys_bp'],
    cat_cols=['edu'],
    num_transform_method='none',
    cat_encoding_method='none'
)
processor.fit(df)
transformed = processor.transform(df)
print("transform 输出的类别列类型:", transformed['edu'].dtype)  # object（字符串）
reconstructed = processor.inverse_transform(transformed)
assert reconstructed['edu'].equals(df['edu'])
print("✓ 'none' 模式测试通过")

In [ ]:
import pandas as pd
import numpy as np
from core.data_processor import TabularDataProcessor  # 请确保类已定义或正确导入

# ---------- 1. 创建带有缺失值的小型数据集 ----------
np.random.seed(42)
n = 10

df_raw = pd.DataFrame({
    'age': [25, 30, np.nan, 45, 52, 33, 41, np.nan, 29, 38],
    'bmi': [22.5, 24.0, 23.1, np.nan, 28.3, 25.7, 26.2, 24.9, np.nan, 27.5],
    'income': [50000, 60000, 55000, 70000, np.nan, 48000, 62000, 58000, 53000, 65000],
    'education': ['high', 'medium', 'low', 'high', np.nan, 'medium', 'low', 'high', 'medium', 'low']
})

print("=== 原始数据（含缺失值）===")
print(df_raw)
print("\n缺失值统计：")
print(df_raw.isnull().sum())

# ---------- 2. 初始化处理器 ----------
# 数值列：age, bmi, income；类别列：education
processor = TabularDataProcessor(
    num_cols=['age', 'bmi', 'income'],
    cat_cols=['education'],
    num_transform_method='quantile',   # 也可用 'standard' 或 'none'
    cat_encoding_method='label',       # 输出类别编码，便于模型使用
    random_state=42
)

# ---------- 3. 拟合并转换（同时完成缺失值插补）----------
processor.fit(df_raw)
df_filled = processor.transform(df_raw)   # 补全后的数据（数值列经分布变换，类别列为整数编码）

print("\n=== 补全后的数据（transform 输出）===")
print(df_filled)
print("\n补全后缺失值统计：", df_filled.isnull().sum().sum())  # 应为0

# ---------- 4. 将补全后的数据还原到原始尺度（可选）----------
df_restored = processor.inverse_transform(df_filled)

print("\n=== 还原到原始尺度的补全数据 ===")
print(df_restored)
print("\n还原后缺失值统计：", df_restored.isnull().sum().sum())

# ---------- 5. 对比原始缺失位置的具体补全值 ----------
print("\n=== 详细插补结果（原始缺失位置 → 补全值）===")
for col in df_raw.columns:
    missing_mask = df_raw[col].isna()
    if missing_mask.any():
        print(f"\n列 '{col}' 原始缺失索引: {df_raw.index[missing_mask].tolist()}")
        print(f"补全值: {df_restored.loc[missing_mask, col].tolist()}")

In [ ]:
import pandas as pd
import numpy as np
from core.data_processor import TabularDataProcessor  # 确保已导入修改后的类

# 1. 创建带缺失值的小数据集
np.random.seed(42)
df_raw = pd.DataFrame({
    'age': [25, 30, np.nan, 45, 52, 33, 41, np.nan, 29, 38],
    'bmi': [22.5, 24.0, 23.1, np.nan, 28.3, 25.7, 26.2, 24.9, np.nan, 27.5],
    'edu': ['high', 'medium', 'low', 'high', np.nan, 'medium', 'low', 'high', 'medium', 'low']
})

print("=== 原始数据 ===")
print(df_raw)
print("\n缺失值统计：")
print(df_raw.isnull().sum())

# 2. 初始化处理器：两个 methods 均为 'none'
processor = TabularDataProcessor(
    num_cols=['age', 'bmi'],
    cat_cols=['edu'],
    num_transform_method='none',
    cat_encoding_method='none',
    random_state=42
)

# 3. 拟合
processor.fit(df_raw)
print("\n=== 拟合完成，各列元信息 ===")
for col, meta in processor.meta.items():
    print(f"{col}: type={meta['type']}, encoder={type(meta['encoder'])}")

# 4. 转换（含插补）
transformed = processor.transform(df_raw)
print("\n=== transform 输出（插补后的数据）===")
print(transformed)
print("\n数据类型：")
print(transformed.dtypes)

# 5. 逆变换
reconstructed = processor.inverse_transform(transformed)
print("\n=== inverse_transform 输出（还原到原始尺度）===")
print(reconstructed)
print("\n数据类型：")
print(reconstructed.dtypes)

# 6. 验证还原准确性
print("\n=== 验证还原结果 ===")
# 数值列
for col in ['age', 'bmi']:
    orig_valid = df_raw[col].dropna()
    rec_valid = reconstructed.loc[orig_valid.index, col]
    if col == 'age':  # 整数列
        assert (rec_valid == orig_valid).all(), f"{col} 整数列还原失败"
        print(f"✓ {col} 整数列完全匹配")
    else:
        assert np.allclose(rec_valid, orig_valid, atol=1e-2), f"{col} 浮点列误差过大"
        print(f"✓ {col} 浮点列在容差内匹配")

# 类别列（注意：原始缺失值被插补后，逆变换会得到插补值，不等于原缺失位置的 NaN）
# 我们只比较非缺失位置是否一致
orig_edu_valid = df_raw['edu'].dropna()
rec_edu_valid = reconstructed.loc[orig_edu_valid.index, 'edu']
assert (rec_edu_valid == orig_edu_valid).all(), "类别列非缺失值还原失败"
print("✓ 类别列非缺失位置完全匹配")

print("\n所有测试通过！")

In [ ]:
import pandas as pd
import numpy as np
from core.data_processor import TabularDataProcessor  # 确保已导入修改后的类

# ==================== 1. 创建带缺失值的测试数据 ====================
np.random.seed(42)
df_raw = pd.DataFrame({
    'age': [25, 30, np.nan, 45, 52, 33, 41, np.nan, 29, 38],
    'bmi': [22.5, 24.0, 23.1, np.nan, 28.3, 25.7, 26.2, 24.9, np.nan, 27.5],
    'edu': ['high', 'medium', 'low', 'high', np.nan, 'medium', 'low', 'high', 'medium', 'low'],
    # 额外列（应被透传，不做任何处理）
    'target': [0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
})

print("=== 原始数据（含缺失值）===")
print(df_raw)
print("\n缺失值统计：")
print(df_raw.isnull().sum())

# ==================== 2. 初始化并拟合处理器 ====================
# 注：num_transform_method 和 cat_encoding_method 对 transform_readable 无影响，
# 因为 readable 内部只进行插补+逆变换，不执行分布映射。
# 但 fit 时仍需指定参数以便正常分析数值结构。
processor = TabularDataProcessor(
    num_cols=['age', 'bmi'],
    cat_cols=['edu'],
    num_transform_method='quantile',   # 任意，不影响 readable
    cat_encoding_method='label',       # 任意，不影响 readable
    random_state=42
)
processor.fit(df_raw)

# ==================== 3. 调用 transform_readable ====================
df_readable = processor.transform_readable(df_raw)

print("\n=== transform_readable 输出（插补后，原始量纲）===")
print(df_readable)
print("\n数据类型：")
print(df_readable.dtypes)

# ==================== 4. 验证结果 ====================
print("\n=== 验证1：缺失值已被补全 ===")
assert df_readable.isnull().sum().sum() == 0, "仍有缺失值未被填充"
print("✓ 所有缺失值已被补全")

print("\n=== 验证2：数值列保持原始量纲和精度 ===")
# age 应为整数
assert df_readable['age'].dtype in ('int64', 'int32'), "age 不是整数类型"
print("✓ age 为整数列")
# bmi 应为浮点数，且小数位数不超过原数据推断的精度（这里原始 bmi 为一位小数）
bmi_decimals = df_readable['bmi'].astype(str).str.split('.').str[1].str.len().fillna(0)
assert (bmi_decimals <= 2).all(), "bmi 小数位数过多"
print("✓ bmi 小数位符合原始精度")

print("\n=== 验证3：类别列为原始字符串 ===")
assert df_readable['edu'].dtype == object, "类别列不是 object 类型"
assert set(df_readable['edu'].unique()).issubset({'high', 'medium', 'low'}), "类别值不正确"
print("✓ 类别列恢复为原始字符串")

print("\n=== 验证4：额外列（target）被直接透传 ===")
assert (df_readable['target'] == df_raw['target']).all(), "额外列被修改"
print("✓ 额外列完全一致")

# ==================== 5. 与 transform + inverse_transform 等价性验证 ====================
# 注意：transform_readable 内部是先编码插补再还原，等价于 transform（无分布变换） + inverse_transform
# 但由于我们 fit 时 num_transform_method='quantile'，普通的 transform 会做分布映射，
# 因此不能直接比较。如果想让二者等价，需将 num_transform_method='none' 重新拟合。
# 所以这里单独验证：用 num_transform_method='none' 的处理器，比较 transform+inverse_transform 与 transform_readable 是否一致。
processor_none = TabularDataProcessor(
    num_cols=['age', 'bmi'],
    cat_cols=['edu'],
    num_transform_method='none',
    cat_encoding_method='label',   # label 编码，但 inverse_transform 会还原
    random_state=42
)
processor_none.fit(df_raw)

# 方式1：transform（无分布变换） + inverse_transform
transformed = processor_none.transform(df_raw)   # 数值列保持原量纲，类别列为编码
reconstructed_from_pipe = processor_none.inverse_transform(transformed)

# 方式2：直接 transform_readable
readable = processor_none.transform_readable(df_raw)

print("\n=== 验证5：transform_readable 等价于 transform+inverse_transform（当 num_method='none' 时）===")
# 比较数值列
for col in ['age', 'bmi']:
    assert np.allclose(reconstructed_from_pipe[col], readable[col], atol=1e-6), f"{col} 不一致"
# 比较类别列（原始字符串）
assert (reconstructed_from_pipe['edu'] == readable['edu']).all(), "类别列不一致"
print("✓ transform_readable 与 pipeline 等效")

# ==================== 6. 查看具体的插补值（可选） ====================
print("\n=== 补全详情（原始缺失位置 -> 补全值）===")
for col in df_raw.columns:
    if col == 'target':
        continue
    missing_mask = df_raw[col].isna()
    if missing_mask.any():
        print(f"{col} 缺失索引 {df_raw.index[missing_mask].tolist()} -> {df_readable.loc[missing_mask, col].tolist()}")

print("\n所有测试通过！ transform_readable 工作正常。")